# Functions and Recursion in Python
### Professional Study Notes

---

**Level:** Beginner → Intermediate  
**Prerequisites:** Variables, data types, conditionals, loops  
**Python version:** 3.8+

## Table of Contents
1. [Functions: Fundamentals](#1.-Functions:-Fundamentals)
2. [Parameters and Arguments](#2.-Parameters-and-Arguments)
3. [Return Values](#3.-Return-Values)
4. [Scope and the LEGB Rule](#4.-Scope-and-the-LEGB-Rule)
5. [Functions as First-Class Objects](#5.-Functions-as-First-Class-Objects)
6. [Lambda, map, filter, sorted](#6.-Lambda,-map,-filter,-sorted)
7. [Closures and Decorators](#7.-Closures-and-Decorators)
8. [Common Pitfalls](#8.-Common-Pitfalls)
9. [Recursion: Fundamentals](#9.-Recursion:-Fundamentals)
10. [Classic Recursive Examples](#10.-Classic-Recursive-Examples)
11. [Optimising Recursion (Memoization)](#11.-Optimising-Recursion-(Memoization))
12. [Recursion Limits and Recursion vs Iteration](#12.-Recursion-Limits-and-Recursion-vs-Iteration)
13. [Best Practices](#13.-Best-Practices)
14. [Practice Exercises](#14.-Practice-Exercises)
15. [Quick Reference Cheat Sheet](#15.-Quick-Reference-Cheat-Sheet)

---
# 1. Functions: Fundamentals

A **function** is a named, reusable block of code that performs a specific task. Functions let you:

- **Avoid repetition** (DRY: *Don't Repeat Yourself*)
- **Organise** code into logical units
- **Test** pieces of logic in isolation
- **Abstract** complexity behind a simple name

### Syntax
```python
def function_name(parameters):
    """Optional docstring describing the function."""
    # body
    return value          # optional
```

| Term | Meaning |
|------|---------|
| **Parameter** | Variable named in the function definition |
| **Argument** | Actual value passed when calling the function |
| **Return value** | Value sent back to the caller (`None` if absent) |

In [ ]:
def greet(name):
    """Return a personalised greeting."""
    return f"Hello, {name}!"

print(greet("Asha"))
print(greet("Ravi"))

### Docstrings
The first string literal in a function is its **docstring**. It is accessible through `help()` or `__doc__` and is the standard way to document behaviour.

In [ ]:
def area_of_circle(radius):
    """
    Calculate the area of a circle.

    Args:
        radius (float): Radius of the circle (must be >= 0).

    Returns:
        float: The area of the circle.
    """
    import math
    return math.pi * radius ** 2

print(area_of_circle(5))
print(area_of_circle.__doc__)

---
# 2. Parameters and Arguments

Python supports several ways to pass data into a function.

## 2.1 Positional and Keyword Arguments

In [ ]:
def describe_pet(animal, name):
    print(f"{name} is a {animal}.")

describe_pet("dog", "Bruno")                 # positional (order matters)
describe_pet(name="Misty", animal="cat")     # keyword (order does not matter)

## 2.2 Default Parameter Values

In [ ]:
def power(base, exponent=2):
    return base ** exponent

print(power(5))        # uses default exponent = 2
print(power(2, 10))    # overrides default

> **Rule:** Parameters with defaults must come **after** parameters without defaults.

## 2.3 Variable-Length Arguments: `*args` and `**kwargs`

- `*args` collects extra **positional** arguments into a **tuple**.
- `**kwargs` collects extra **keyword** arguments into a **dict**.

In [ ]:
def total(*args):
    print("args =", args)
    return sum(args)

print(total(1, 2, 3, 4))

def show_profile(**kwargs):
    for key, value in kwargs.items():
        print(f"{key}: {value}")

show_profile(name="Asha", city="Bhopal", role="Engineer")

## 2.4 Unpacking Arguments with `*` and `**`

In [ ]:
def add(a, b, c):
    return a + b + c

nums = [1, 2, 3]
print(add(*nums))                 # unpack list -> positional args

params = {"a": 10, "b": 20, "c": 30}
print(add(**params))              # unpack dict -> keyword args

## 2.5 Keyword-Only and Positional-Only Parameters

- Parameters **after** `*` are keyword-only.
- Parameters **before** `/` are positional-only (Python 3.8+).

In [ ]:
def connect(host, port, *, timeout=30, secure=True):
    return f"{host}:{port} timeout={timeout} secure={secure}"

print(connect("localhost", 8080, timeout=10))
# connect("localhost", 8080, 10)  # TypeError: timeout must be passed by keyword

def square(x, /):
    return x * x

print(square(4))
# square(x=4)  # TypeError: x is positional-only

### Full Parameter Order
```
def f(pos_only, /, normal, *args, kw_only, **kwargs):
```

---
# 3. Return Values

- A function returns `None` implicitly if no `return` is executed.
- `return` immediately exits the function.
- Multiple values are returned as a **tuple**, which can be unpacked.

In [ ]:
def min_max(values):
    return min(values), max(values)   # returns a tuple

low, high = min_max([7, 2, 9, 4])
print(low, high)

def no_return():
    pass

print(no_return())    # None

### Early Return (Guard Clauses)

In [ ]:
def safe_divide(a, b):
    if b == 0:
        return None          # guard clause: handle the edge case first
    return a / b

print(safe_divide(10, 2))
print(safe_divide(10, 0))

---
# 4. Scope and the LEGB Rule

Python resolves names in this order:

| Level | Meaning |
|-------|---------|
| **L**ocal | Inside the current function |
| **E**nclosing | In enclosing (outer) functions |
| **G**lobal | At module level |
| **B**uilt-in | Python's built-in names (`len`, `print`, ...) |

In [ ]:
x = "global"

def outer():
    x = "enclosing"

    def inner():
        x = "local"
        print("inner sees:", x)

    inner()
    print("outer sees:", x)

outer()
print("module sees:", x)

### `global` and `nonlocal`

In [ ]:
counter = 0

def increment_global():
    global counter          # modify the module-level variable
    counter += 1

increment_global()
increment_global()
print(counter)

def make_counter():
    count = 0
    def inc():
        nonlocal count      # modify the enclosing variable
        count += 1
        return count
    return inc

c = make_counter()
print(c(), c(), c())

> **Best practice:** Avoid `global`. Prefer passing arguments and returning values.

---
# 5. Functions as First-Class Objects

In Python, functions are objects. You can:

- assign them to variables
- store them in data structures
- pass them as arguments
- return them from other functions

In [ ]:
def shout(text):
    return text.upper() + "!"

def whisper(text):
    return text.lower() + "..."

def speak(style, message):
    return style(message)          # style is a function

print(speak(shout, "Hello"))
print(speak(whisper, "HELLO"))

operations = {"shout": shout, "whisper": whisper}
print(operations["shout"]("dispatch table"))

---
# 6. Lambda, map, filter, sorted

A **lambda** is a small anonymous function limited to a single expression.

```python
lambda arguments: expression
```

In [ ]:
square = lambda x: x ** 2
print(square(6))

numbers = [1, 2, 3, 4, 5, 6]

print(list(map(lambda n: n * n, numbers)))         # transform each item
print(list(filter(lambda n: n % 2 == 0, numbers))) # keep items that pass

students = [("Asha", 88), ("Ravi", 72), ("Meera", 95)]
print(sorted(students, key=lambda s: s[1], reverse=True))  # custom sort key

> **Tip:** List comprehensions are often more readable than `map`/`filter`:  
> `[n * n for n in numbers if n % 2 == 0]`

---
# 7. Closures and Decorators

## 7.1 Closures
A **closure** is an inner function that remembers variables from its enclosing scope even after the outer function has finished.

In [ ]:
def multiplier(factor):
    def multiply(x):
        return x * factor     # 'factor' is remembered
    return multiply

double = multiplier(2)
triple = multiplier(3)
print(double(10), triple(10))

## 7.2 Decorators
A **decorator** wraps a function to extend its behaviour without modifying its code. The `@decorator` syntax is shorthand for `func = decorator(func)`.

In [ ]:
import time
from functools import wraps

def timer(func):
    @wraps(func)                       # preserves name and docstring
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__} took {elapsed:.6f}s")
        return result
    return wrapper

@timer
def slow_sum(n):
    """Sum numbers from 0 to n-1."""
    return sum(range(n))

print(slow_sum(1_000_000))
print(slow_sum.__name__)

---
# 8. Common Pitfalls

## 8.1 Mutable Default Arguments
Default values are evaluated **once**, when the function is defined, not on each call.

In [ ]:
# BAD
def add_item_bad(item, items=[]):
    items.append(item)
    return items

print(add_item_bad("a"))
print(add_item_bad("b"))     # surprise: ['a', 'b']

# GOOD
def add_item_good(item, items=None):
    if items is None:
        items = []
    items.append(item)
    return items

print(add_item_good("a"))
print(add_item_good("b"))    # ['b']

## 8.2 Forgetting to `return`
A function that only `print`s returns `None`. Use `return` when the result is needed elsewhere.

## 8.3 Modifying Arguments In Place
Lists and dicts are passed by object reference, so mutations inside a function affect the caller's object.

In [ ]:
def append_zero(lst):
    lst.append(0)

data = [1, 2]
append_zero(data)
print(data)          # [1, 2, 0]: the original changed

---
# 9. Recursion: Fundamentals

**Recursion** is a technique where a function calls **itself** to solve a smaller instance of the same problem.

## Two Essential Components

1. **Base case:** the simplest input, answered directly, which stops the recursion.
2. **Recursive case:** reduces the problem and calls the function on the smaller input.

> Without a base case, recursion never terminates and raises `RecursionError`.

## Template
```python
def solve(problem):
    if problem is simple:            # base case
        return direct_answer
    smaller = reduce(problem)        # make progress toward the base case
    return combine(solve(smaller))   # recursive case
```

## Example: Factorial
`n! = n × (n-1)!` with `0! = 1`

In [ ]:
def factorial(n):
    if n < 0:
        raise ValueError("n must be non-negative")
    if n == 0:                       # base case
        return 1
    return n * factorial(n - 1)      # recursive case

print(factorial(5))     # 120
print(factorial(10))    # 3628800

## The Call Stack
Each call is pushed onto the **call stack** and waits for its inner call to return. Tracing `factorial(4)`:

```
factorial(4)
 = 4 * factorial(3)
 = 4 * (3 * factorial(2))
 = 4 * (3 * (2 * factorial(1)))
 = 4 * (3 * (2 * (1 * factorial(0))))
 = 4 * (3 * (2 * (1 * 1)))      <- base case reached, stack unwinds
 = 24
```

The function below prints the winding and unwinding of the stack.

In [ ]:
def factorial_trace(n, depth=0):
    indent = "  " * depth
    print(f"{indent}-> factorial({n})")
    if n == 0:
        print(f"{indent}<- returns 1")
        return 1
    result = n * factorial_trace(n - 1, depth + 1)
    print(f"{indent}<- returns {result}")
    return result

factorial_trace(4)

---
# 10. Classic Recursive Examples

## 10.1 Sum of a List

In [ ]:
def list_sum(nums):
    if not nums:                         # base case: empty list
        return 0
    return nums[0] + list_sum(nums[1:])  # first element + sum of the rest

print(list_sum([3, 5, 7, 9]))

## 10.2 Reverse a String and Palindrome Check

In [ ]:
def reverse(s):
    if len(s) <= 1:
        return s
    return reverse(s[1:]) + s[0]

def is_palindrome(s):
    s = "".join(ch.lower() for ch in s if ch.isalnum())
    def check(lo, hi):
        if lo >= hi:
            return True
        if s[lo] != s[hi]:
            return False
        return check(lo + 1, hi - 1)
    return check(0, len(s) - 1)

print(reverse("recursion"))
print(is_palindrome("A man, a plan, a canal: Panama"))
print(is_palindrome("Python"))

## 10.3 Sum of Digits and Fast Exponentiation

In [ ]:
def sum_digits(n):
    if n < 10:
        return n
    return n % 10 + sum_digits(n // 10)

def fast_power(base, exp):
    """Exponentiation by squaring: O(log n) instead of O(n)."""
    if exp == 0:
        return 1
    half = fast_power(base, exp // 2)
    if exp % 2 == 0:
        return half * half
    return half * half * base

print(sum_digits(98765))
print(fast_power(2, 30))

## 10.4 Binary Search (Divide and Conquer)
Halve the search space on every call. Requires a **sorted** list. Time: O(log n).

In [ ]:
def binary_search(arr, target, lo=0, hi=None):
    if hi is None:
        hi = len(arr) - 1
    if lo > hi:
        return -1                          # base case: not found
    mid = (lo + hi) // 2
    if arr[mid] == target:
        return mid                         # base case: found
    if arr[mid] < target:
        return binary_search(arr, target, mid + 1, hi)
    return binary_search(arr, target, lo, mid - 1)

data = [2, 5, 8, 12, 16, 23, 38, 56, 72, 91]
print(binary_search(data, 23))   # 5
print(binary_search(data, 7))    # -1

## 10.5 Flatten a Nested List
Recursion is a natural fit for **nested or tree-shaped data**.

In [ ]:
def flatten(items):
    flat = []
    for item in items:
        if isinstance(item, list):
            flat.extend(flatten(item))     # recurse into sublists
        else:
            flat.append(item)
    return flat

print(flatten([1, [2, [3, 4]], [[5], 6]]))

## 10.6 Tower of Hanoi
Move *n* disks from peg A to peg C using peg B, never placing a larger disk on a smaller one. Minimum moves: **2ⁿ − 1**.

In [ ]:
def hanoi(n, source, target, spare, moves=None):
    if moves is None:
        moves = []
    if n == 1:
        moves.append((source, target))
        return moves
    hanoi(n - 1, source, spare, target, moves)   # move n-1 disks out of the way
    moves.append((source, target))               # move the largest disk
    hanoi(n - 1, spare, target, source, moves)   # move n-1 disks onto it
    return moves

result = hanoi(3, "A", "C", "B")
print(len(result), "moves")
for step, (src, dst) in enumerate(result, 1):
    print(f"{step}. {src} -> {dst}")

## 10.7 Generating Permutations (Backtracking)

In [ ]:
def permutations(items):
    if len(items) <= 1:
        return [items]
    result = []
    for i, ch in enumerate(items):
        rest = items[:i] + items[i + 1:]
        for perm in permutations(rest):
            result.append(ch + perm)
    return result

print(permutations("abc"))

## 10.8 Traversing a Directory-Like Tree

In [ ]:
def print_tree(node, depth=0):
    for name, children in node.items():
        print("  " * depth + name)
        if isinstance(children, dict):
            print_tree(children, depth + 1)

filesystem = {
    "project": {
        "src": {"main.py": None, "utils.py": None},
        "tests": {"test_main.py": None},
        "README.md": None,
    }
}
print_tree(filesystem)

---
# 11. Optimising Recursion (Memoization)

## The Problem: Naive Fibonacci
`fib(n) = fib(n-1) + fib(n-2)` recomputes the same values repeatedly, giving **O(2ⁿ)** time.

In [ ]:
import time

def fib_naive(n):
    if n < 2:
        return n
    return fib_naive(n - 1) + fib_naive(n - 2)

start = time.perf_counter()
print(fib_naive(30))
print(f"naive: {time.perf_counter() - start:.3f}s")

## The Fix: Memoization
**Memoization** caches results of previous calls. Time drops to **O(n)**.

Python provides `functools.lru_cache` (or `functools.cache` in 3.9+).

In [ ]:
from functools import lru_cache

@lru_cache(maxsize=None)
def fib_memo(n):
    if n < 2:
        return n
    return fib_memo(n - 1) + fib_memo(n - 2)

start = time.perf_counter()
print(fib_memo(30))
print(f"memoized: {time.perf_counter() - start:.6f}s")
print(fib_memo(80))
print(fib_memo.cache_info())

### Manual Memoization with a Dictionary

In [ ]:
def fib_dict(n, memo=None):
    if memo is None:
        memo = {}
    if n in memo:
        return memo[n]
    if n < 2:
        return n
    memo[n] = fib_dict(n - 1, memo) + fib_dict(n - 2, memo)
    return memo[n]

print(fib_dict(50))

---
# 12. Recursion Limits and Recursion vs Iteration

## 12.1 The Recursion Limit
Python caps recursion depth (default about 1000) to protect against stack overflow. Python does **not** perform tail-call optimisation.

In [ ]:
import sys
print("Default limit:", sys.getrecursionlimit())

def endless(n):
    return endless(n + 1)     # no base case!

try:
    endless(0)
except RecursionError as e:
    print("RecursionError:", e)

> Use `sys.setrecursionlimit(n)` sparingly. Raising it too high can crash the interpreter. If depth is large, convert to iteration.

## 12.2 Recursion vs Iteration

In [ ]:
def factorial_iter(n):
    result = 1
    for i in range(2, n + 1):
        result *= i
    return result

print(factorial_iter(5))
print(len(str(factorial_iter(1000))), "digits (works fine, no stack issue)")

| Aspect | Recursion | Iteration |
|--------|-----------|-----------|
| Readability | Excellent for trees, graphs, divide-and-conquer | Excellent for simple repetition |
| Memory | Uses call stack: O(depth) | Typically O(1) extra |
| Speed | Function-call overhead | Usually faster |
| Risk | `RecursionError` on deep inputs | None |
| Best for | Hierarchical or self-similar problems | Linear, flat problems |

### Types of Recursion
- **Direct:** function calls itself (`factorial`)
- **Indirect (mutual):** `A` calls `B`, `B` calls `A`
- **Tail:** recursive call is the last operation (not optimised in Python)
- **Tree:** multiple recursive calls per invocation (`fib_naive`, `hanoi`)

In [ ]:
# Indirect (mutual) recursion
def is_even(n):
    return True if n == 0 else is_odd(n - 1)

def is_odd(n):
    return False if n == 0 else is_even(n - 1)

print(is_even(10), is_odd(7))

---
# 13. Best Practices

### Functions
1. **Single responsibility:** one function, one job.
2. **Descriptive names:** use `snake_case` verbs, e.g. `calculate_tax`.
3. **Keep them short:** aim for a function that fits on one screen.
4. **Write docstrings** and add **type hints** for clarity.
5. **Prefer pure functions** (no side effects) where possible; they are easier to test.
6. **Avoid mutable default arguments.**
7. **Avoid `global`** state; pass data explicitly.

### Recursion
1. Always define a **base case** first.
2. Ensure each call **makes progress** toward the base case.
3. Use **memoization** when subproblems overlap.
4. Prefer **iteration** when depth could be very large.
5. Validate inputs (e.g. reject negative `n`).
6. Test with **small inputs and edge cases** (0, 1, empty, single element).

### Type Hints Example

In [ ]:
from typing import List, Optional

def find_max(values: List[int]) -> Optional[int]:
    """Return the largest value, or None if the list is empty."""
    if not values:
        return None
    return max(values)

print(find_max([3, 9, 2]))
print(find_max([]))

---
# 14. Practice Exercises

Try each exercise yourself before checking the solutions in the next section.

**Functions**
1. Write `is_prime(n)` that returns `True` if `n` is prime.
2. Write `stats(*numbers)` that returns the mean, minimum and maximum.
3. Write a decorator `@log_calls` that prints the function name and arguments on each call.

**Recursion**
4. Write `count_down(n)` that prints `n` down to 1 recursively.
5. Write `gcd(a, b)` using Euclid's algorithm recursively.
6. Write `power_set(items)` that returns all subsets of a list.
7. Write `depth(nested)` that returns the maximum nesting depth of a list.
8. Write `merge_sort(arr)` recursively.

## Solutions

In [ ]:
# 1. is_prime
def is_prime(n):
    if n < 2:
        return False
    for i in range(2, int(n ** 0.5) + 1):
        if n % i == 0:
            return False
    return True

# 2. stats
def stats(*numbers):
    return sum(numbers) / len(numbers), min(numbers), max(numbers)

# 3. log_calls decorator
from functools import wraps
def log_calls(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__} with args={args}, kwargs={kwargs}")
        return func(*args, **kwargs)
    return wrapper

@log_calls
def add(a, b):
    return a + b

print([n for n in range(20) if is_prime(n)])
print(stats(4, 8, 15, 16, 23, 42))
print(add(2, 3))

In [ ]:
# 4. count_down
def count_down(n):
    if n == 0:
        return
    print(n, end=" ")
    count_down(n - 1)

# 5. gcd
def gcd(a, b):
    return a if b == 0 else gcd(b, a % b)

# 6. power_set
def power_set(items):
    if not items:
        return [[]]
    rest = power_set(items[1:])
    return rest + [[items[0]] + subset for subset in rest]

# 7. depth
def depth(nested):
    if not isinstance(nested, list):
        return 0
    return 1 + max((depth(x) for x in nested), default=0)

# 8. merge_sort
def merge_sort(arr):
    if len(arr) <= 1:
        return arr
    mid = len(arr) // 2
    left, right = merge_sort(arr[:mid]), merge_sort(arr[mid:])
    merged, i, j = [], 0, 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            merged.append(left[i]); i += 1
        else:
            merged.append(right[j]); j += 1
    return merged + left[i:] + right[j:]

count_down(5); print()
print(gcd(48, 18))
print(power_set([1, 2, 3]))
print(depth([1, [2, [3, [4]]]]))
print(merge_sort([38, 27, 43, 3, 9, 82, 10]))

---
# 15. Quick Reference Cheat Sheet

### Function Anatomy
```python
def name(pos, /, normal, default=1, *args, kw_only, **kwargs) -> ReturnType:
    """Docstring."""
    return value
```

### Key Concepts
| Concept | One-line summary |
|---------|------------------|
| `return` | Sends a value back and exits the function |
| `*args` / `**kwargs` | Collect extra positional / keyword arguments |
| LEGB | Local → Enclosing → Global → Built-in name lookup |
| `nonlocal` / `global` | Rebind an enclosing / module-level variable |
| Lambda | Anonymous single-expression function |
| Closure | Inner function that remembers outer variables |
| Decorator | Function that wraps another function |
| Recursion | A function that calls itself on a smaller problem |
| Base case | Condition that stops the recursion |
| Memoization | Caching results to avoid recomputation (`lru_cache`) |
| Recursion limit | About 1000 frames by default; no tail-call optimisation |

### Recursion Checklist
- [ ] Is there a clear **base case**?
- [ ] Does every recursive call move **closer** to the base case?
- [ ] Are there **overlapping subproblems**? If yes, memoize.
- [ ] Could the depth exceed **~1000**? If yes, use iteration or an explicit stack.

---
*End of notes. Happy coding!*